# Mamba SOH — Long-sequence training (L=4096) v2 — GH-10

Train `MambaSOHPredictor` v2.0 ở chuỗi dài (warmup 256→4096 + gradient accumulation + attention pooling) trên Kaggle GPU.

**Trước khi chạy:**
1. Settings → Accelerator → **GPU P100/T4**
2. + Add Data → dataset NASA chứa `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/*.csv`
3. (repo private) Add-ons → Secrets → tạo `GITHUB_TOKEN` = GitHub PAT

**v2.0 improvements (so với v1.x MAE ceiling 2.24%):**
| Thay đổi | Chi tiết |
|---|---|
| +2 features (8 total) | IC curve (dQ/dV, SOH indicator) + discharge progress (phase channel) |
| `PatchDegradationEncoder` | Local RMS/P2P/std/kurtosis per 16-step patch → per-token local context |
| 2-layer FiLM (SiLU) | Deeper feature conditioning vs single Linear |
| `CosineAnnealingWarmRestarts` | Periodic LR restarts escape sharp minima (replaces ReduceLROnPlateau) |
| `SmoothL1Loss(beta=0.02)` | Clamp gradient for >2% residuals; reduce label noise impact |
| Discharge-weighted attention | Last channel (discharge_progress) steers attn toward end-of-discharge |
| `LONG_SEQ_STRIDE=64` | ~4400 train windows (2× v1.x) |
| Scheduler reset at final stage | v1.x bug: warmup transitions reduced LR before final stage started |

> ⚠️ `mode="reduce-overhead"` (CUDA Graphs) **không dùng được khi training** — dùng `mode="default"`.
> `scaler_long.pkl` (8 features) được tạo bởi `preprocess_long.py` — độc lập với `scaler.pkl` (6 features).

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: bật GPU ở Settings -> Accelerator -> GPU P100/T4')

## 2 — Clone branch GH-10

In [ ]:
import subprocess
BRANCH  = 'feat/spectral_kurtosis'   # branch chứa dt_rank fix + --compile/--benchmark + parallel preprocess
REPO    = '/kaggle/working/ai-module'
URL_PUB = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thử public clone:', e)
    url = URL_PUB
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', url, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL_PUB])  # xoá token khỏi remote
print('Branch:', subprocess.check_output(['git','-C',REPO,'branch','--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git','-C',REPO,'log','-1','--oneline']).decode().strip())

## 3 — Dependencies (torch đã có sẵn trên Kaggle)

In [ ]:
%pip install -q scipy scikit-learn joblib pandas
import scipy, sklearn; print('scipy', scipy.__version__, '| sklearn', sklearn.__version__)

In [ ]:
## 3b — (Optional) Install official mamba-ssm CUDA backend
# Nếu thành công: train nhanh hơn ~3-5x, ít VRAM hơn ở L=4096.
# Nếu fail (version mismatch): tự fallback về pure-PyTorch — không cần làm gì thêm.
import subprocess, sys, torch

USE_OFFICIAL_MAMBA = True   # đổi False nếu muốn dùng pure-PyTorch

if USE_OFFICIAL_MAMBA and torch.cuda.is_available():
    print("Installing causal-conv1d + mamba-ssm (build ~5-10 phút lần đầu)...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "causal-conv1d>=1.1.0"], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mamba-ssm"],            capture_output=True)
    try:
        import mamba_ssm
        print(f"mamba-ssm {mamba_ssm.__version__} OK — official CUDA Mamba sẽ được dùng (3-5x nhanh hơn)")
    except ImportError:
        print("mamba-ssm install failed — sẽ fallback về pure-PyTorch MambaBlock tự động")
        USE_OFFICIAL_MAMBA = False
else:
    print("Skipping mamba-ssm install (USE_OFFICIAL_MAMBA=False hoặc không có GPU)")
    USE_OFFICIAL_MAMBA = False

print(f"USE_OFFICIAL_MAMBA = {USE_OFFICIAL_MAMBA}")

## 4 — Tìm NASA dataset + vào repo

In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [f for f in subprocess.check_output(['find','/kaggle/input','-name','metadata.csv']).decode().splitlines() if f]
assert found, 'Khong thay metadata.csv — + Add Data dataset NASA cleaned_dataset'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET    :', DATASET)
print('has data/  :', os.path.isdir(f'{DATASET}/data'))
print('cwd        :', os.getcwd())
print('scaler.pkl :', os.path.isfile('models/weights/scaler.pkl'), '(committed, preprocess_long reuse)')

## 5 — Preprocess (ghép cycle → chuỗi 4096)

Tạo `data/processed_long/{train,val,test}.pt` + `models/weights/feature_scaler_long.pkl`.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_long.py --data-dir "{DATASET}" --output-dir data/processed_long

## 6 — Smoke test (1 epoch/stage) — kiểm tra pipeline trước

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
# Smoke test: 1 epoch/stage — verify pipeline trước khi chạy full
!python scripts/train.py --long --stage-epochs 1 --final-epochs 1 --micro-batch 8 --benchmark

## 7 — Full training

`--patch-size 16 --patch-stride 16`: compresses L=4096 → 256 tokens (16×) before Mamba.
- Reduces VRAM ~5-6× (573 MB → ~95 MB inference) + training time ~4-5×
- Each 16-step patch captures a local sub-cycle pattern; Mamba models degradation across 256 patch tokens
- Inspired by MambaDecomp_P16 (PatchTST, NeurIPS 2023) — same accuracy, much faster

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
# v2.0 full training:
#   --patch-size 16 --patch-stride 16 : L=4096 → 256 tokens (16×, PatchDegradationEncoder active)
#   --weighted-loss                   : SmoothL1 + upweight near-EOL samples (SOH approaching 80%)
#   --official-mamba                  : CUDA mamba_ssm if installed (cell 3b), falls back to pure-PyTorch
official_flag = '--official-mamba' if USE_OFFICIAL_MAMBA else ''
!python scripts/train.py --long {official_flag} --compile --benchmark --num-workers 4 \
    --patch-size 16 --patch-stride 16 --weighted-loss

## 8 — Kết quả + đóng gói artifact

In [ ]:
import os, glob, shutil, torch
os.chdir('/kaggle/working/ai-module')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    print('Log:', logs[-1]); print('-'*50)
    !grep -E "Test MAE|Test RMSE|\[stage|Saved long|Patch:|Final stage" "{logs[-1]}"
ck = 'models/weights/soh_mamba_long_v1.0.pth'
if os.path.isfile(ck):
    c = torch.load(ck, map_location='cpu', weights_only=False)
    print('-'*50)
    ps, ss = c.get('patch_size', 1), c.get('patch_stride', 1)
    num_tokens = (c['seq_len'] - ps) // ss + 1 if ps > 1 else c['seq_len']
    print(f"seq_len={c['seq_len']} patch={ps}s{ss} → {num_tokens} tokens | pooling={c['pooling']}")
    print(f"Test MAE={c['test_mae']:.4f}%  RMSE={c['test_rmse']:.4f}%")
# package artifact to download
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in ['models/weights/soh_mamba_long_v1.0.pth', 'models/weights/feature_scaler_long.pkl']:
    if os.path.isfile(f): shutil.copy2(f, '/kaggle/working/out/'); print('copied', f)
shutil.make_archive('/kaggle/working/mamba_long_artifacts', 'zip', '/kaggle/working/out')
print('\nDownload: Output tab -> mamba_long_artifacts.zip')
print('Commit 2 file vao branch GH-10 (PR #11) + dien MAE vao PR.')

## Nếu MAE > 2%

**Các cải thiện đã tích hợp (stride=64, scheduler reset, stage_epochs=5)** nhằm phá vỡ ceiling 2.24%:
- `LONG_SEQ_STRIDE = 64` (giảm từ 128) → ~4400 train windows (2×)
- Final stage reset LR + fresh `ReduceLROnPlateau` → tránh LR bị giảm do spike khi chuyển stage warmup
- `stage_epochs=5` (tăng từ 3) → model học ổn định hơn ở mỗi độ dài trước khi chuyển sang dài hơn

Nếu SAU KHI áp các fix trên MAE vẫn > 2%, thử **hạ L=2048**:
```python
import re, pathlib
p = pathlib.Path('/kaggle/working/ai-module/src/core/config.py')
p.write_text(re.sub(r'LONG_SEQ_LEN\s*=\s*4096', 'LONG_SEQ_LEN    = 2048', p.read_text()))
```

Với L=2048 + `--compile --benchmark`, full training hoàn thành trong **~25-40 phút** (vs 2-3h với L=4096).